In [1]:
from dg2cd.losses import *
import torch

In [2]:
# Lets test the function with some dummy data
if __name__ == "__main__":
    # Create dummy features and labels
    features = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
    labels = torch.tensor([0, 1, 0, 1])
    num_classes = 3

    # Compute class prototypes
    prototypes, present = class_prototypes(features, labels, num_classes)

    print("Prototypes:\n", prototypes)
    print("Present classes:\n", present)

Prototypes:
 tensor([[3., 4.],
        [5., 6.],
        [0., 0.]])
Present classes:
 tensor([ True,  True, False])


In [3]:
import torch

features = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0],
])

labels = torch.tensor([0, 1, 0, 1])

loss_internal = supervised_prototype_contrastive_loss(
    features=features,
    labels=labels,
    num_classes=2,
    temperature=1.0,
    reduction="mean",
)

prototypes = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
])

loss_external = supervised_prototype_contrastive_loss(
    features=features,
    labels=labels,
    prototypes=prototypes,
    temperature=1.0,
    reduction="mean",
)

expected = torch.tensor(0.3132617)

print("Internal:", loss_internal.item())
print("External:", loss_external.item())
print("Expected:", expected.item())

assert torch.allclose(loss_internal, expected, atol=1e-6)
assert torch.allclose(loss_external, expected, atol=1e-6)
assert torch.allclose(loss_internal, loss_external, atol=1e-6)

print("All tests passed")

Internal: 0.31326165795326233
External: 0.31326165795326233
Expected: 0.3132616877555847
All tests passed


In [2]:
import torch
from dg2cd.losses import unsupervised_contrastive_loss

torch.manual_seed(0)
B, D = 8, 16

# 1. Views identical -> positive similarity is maximal -> loss near its floor.
z = torch.randn(B, D)
print("identical views:", unsupervised_contrastive_loss(z, z).item())

# 2. Views unrelated -> no usable signal -> loss approaches log(2B-1).
import math
z1, z2 = torch.randn(B, D), torch.randn(B, D)
print("random views:   ", unsupervised_contrastive_loss(z1, z2).item())
print("log(2B-1) =     ", math.log(2 * B - 1))

# 3. Gradients reach the encoder.
z1 = torch.randn(B, D, requires_grad=True)
unsupervised_contrastive_loss(z1, torch.randn(B, D)).backward()
print("grad finite:", torch.isfinite(z1.grad).all().item(), "| nonzero:", (z1.grad.abs().sum() > 0).item())

identical views: 0.0005668401718139648
random views:    13.967005729675293
log(2B-1) =      2.70805020110221
grad finite: True | nonzero: True


In [1]:
import math
import torch
from dg2cd.losses import open_set_adversarial_loss
from dg2cd.models import grad_reverse

# --- Eq. 8: minimum sits exactly at p(unknown) = alpha = 0.5 ---
def loss_at(p_unknown):
    # two known classes sharing (1 - p) equally, plus the unknown class
    p = torch.tensor([[(1 - p_unknown) / 2, (1 - p_unknown) / 2, p_unknown]])
    return open_set_adversarial_loss(torch.log(p)).item()   # log-probs are valid logits

for p in [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    print(f"p(unknown)={p:>4}  L_adv={loss_at(p):.4f}")
print("expected minimum log(2) =", math.log(2))

# --- no NaN even at the extremes the encoder is pushing toward ---
extreme = torch.tensor([[-60.0, -60.0, 60.0]])          # p(unknown) = 1.0 in float32
print("extreme logits ->", open_set_adversarial_loss(extreme).item(), "(finite, no NaN)")

# --- GRL: forward identity, backward negated ---
x = torch.randn(4, 3, requires_grad=True)
grad_reverse(x, 1.0).sum().backward()
print("GRL grad (expect all -1):", x.grad.unique().tolist())

x = torch.randn(4, 3, requires_grad=True)
(x * 1.0).sum().backward()
print("plain grad (expect all +1):", x.grad.unique().tolist())

p(unknown)=0.01  L_adv=2.3076
p(unknown)= 0.1  L_adv=1.2040
p(unknown)= 0.3  L_adv=0.7803
p(unknown)= 0.5  L_adv=0.6931
p(unknown)= 0.7  L_adv=0.7803
p(unknown)= 0.9  L_adv=1.2040
p(unknown)=0.99  L_adv=2.3076
expected minimum log(2) = 0.6931471805599453
extreme logits -> 59.65342712402344 (finite, no NaN)
GRL grad (expect all -1): [-1.0]
plain grad (expect all +1): [1.0]


In [1]:
import torch
from dg2cd.losses import confidence_margin_loss

def loss_for(probs):
    """probs: last entry is the unknown class."""
    p = torch.tensor([probs])
    return confidence_margin_loss(torch.log(p)).item()

cases = {
    "confident known   [0.90, 0.05 | 0.05]": [0.90, 0.05, 0.05],
    "confident novel   [0.05, 0.05 | 0.90]": [0.05, 0.05, 0.90],
    "exactly at margin [0.85, 0.00 | 0.15]": [0.85, 0.001, 0.149],
    "mildly unsure     [0.50, 0.10 | 0.40]": [0.50, 0.10, 0.40],
    "fully uniform     [0.33, 0.33 | 0.33]": [1/3, 1/3, 1/3],
}
for name, p in cases.items():
    print(f"{name}  ->  L_margin = {loss_for(p):.4f}")

# Symmetry: confident-known and confident-novel must cost the same.
print("\nsymmetric:", loss_for([0.9, 0.05, 0.05]) == loss_for([0.05, 0.05, 0.9]))

# Hinge: samples past the margin get zero gradient and cannot be pushed further.
x = torch.log(torch.tensor([[0.90, 0.05, 0.05]])).requires_grad_(True)
confidence_margin_loss(x).backward()
print("grad when satisfied (expect all 0):", x.grad.abs().sum().item())

y = torch.log(torch.tensor([[1/3, 1/3, 1/3]])).requires_grad_(True)
confidence_margin_loss(y).backward()
print("grad when ambiguous (expect > 0):  ", y.grad.abs().sum().item())

confident known   [0.90, 0.05 | 0.05]  ->  L_margin = 0.0000
confident novel   [0.05, 0.05 | 0.90]  ->  L_margin = 0.0000
exactly at margin [0.85, 0.00 | 0.15]  ->  L_margin = 0.0000
mildly unsure     [0.50, 0.10 | 0.40]  ->  L_margin = 0.6000
fully uniform     [0.33, 0.33 | 0.33]  ->  L_margin = 0.7000

symmetric: True
grad when satisfied (expect all 0): 0.0
grad when ambiguous (expect > 0):   0.0
